In [ ]:
import numpy as np
import scipy as sp

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import animation

from pyfade import Filter_type, get_filter, get_MP, plot_multiple, get_KDP, apply_filter, get_signal_decomp, get_MP_from_wavelets, wavelet_KDP

ImportError: cannot import name 'print_multiple' from 'pyfade' (c:\Users\Heitor\anaconda3\Lib\site-packages\pyfade\__init__.py)

## Example

### Data generation

This section presents an example for stumpy usage, based on the Matrix Profile papers from Professor Eamon Keogh.

Each synthetic signal is composed of three components:

$$y_i = b_i + h_i + s_i$$

$b_i$ is a low frequency component

$h_i$ is a high frequency component

$s_i$ is a random noise added throughout the whole signal

In [ ]:
# Generating multi-dimensional time series
C = 5                                   # Dimension (number of signals)
N = 300                                 # Number of measurements
sigma = np.array([1, 2, 1, 2, 1])*0.1   # Standard deviation of noise from each signal


# Generating the components of the time series
ti = np.arange(N)
y0 = np.zeros((C,N), dtype=float)
bi = np.zeros((C,N), dtype=float)
hi = np.zeros((C,N), dtype=float)
si = np.zeros((C,N), dtype=float)



for i in range(C):
    bi[i,:] = np.sin(0.2*ti)
    si[i,:] = np.random.normal(0,sigma[i],N)
    hi[i,:] = 0.3*np.sin(2*ti) + 0.2*np.sin(5*ti) + 0.8*np.sin(1*ti)
    y0[i,:] = bi[i,:] + hi[i,:] + si[i,:]

# Plotting the figures
fig, axs = plt.subplots(C, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,1.5*C])

# So it doesn't break if C == 1
if C == 1: 
    axs = [axs]  

# Plot figure
for i in range(C):
    axs[i].plot(ti,y0[i,:]);
    axs[i].set_ylabel(f'Signal {i+1}')
    axs[i].grid(True)

axs[i].set_xlabel('Timestamp');


### Noise insertion

After generating the "proper" data, we insert anomalies in it. This is done by including random noise at certain locations

In [ ]:
# Size of the anomalies
size = N//10 

# Anomaly location (# signals with anomaly, start, size)
anomalies = [(2,int(0.2*N),size,[]), 
             (3,int(0.5*N),size,[]),
             (1,int(0.8*N),size,[])]

y = y0.copy()


for o in range(len(anomalies)):

    # Getting information, choosing faulty signals and saving this info
    num_signals,start,size,_  = anomalies[o]
    series = np.random.default_rng().choice(C,size=num_signals,replace=False)
    anomalies[o][3].extend(series)
    print(series, i)

    t_an = np.linspace(0,4*size,size)
    for s in series:
        y[s,start:start+size] += 2*np.random.rand(size)
        #y[s,start:start+size] += 0.4*np.sin(t_an)


# Plot graphs
max_y = 2.5
fig, axs = print_multiple(y, height=1, width=7, ylimits=[-max_y,max_y], ylabel='Signal', xlabel='Timestamp')

# Plot anomalies
for _,start,size,series in anomalies:
    for s in series:
        rect = Rectangle((start, -max_y), size, 2*max_y, facecolor='lightgrey')
        axs[s].add_patch(rect)


### Multi-level wavelet decomposition

To find the anomalies, a multi-level wavelet decomposition is done.

In [ ]:
filtering_on = True

import pandas as pd
from pyfade import get_MP_from_wavelets

lvl = 3
sig = 3
yd = pd.Series(y[sig,:])

wavelet = 'db2'
sigs,_= get_signal_decomp(yd,wavelet=wavelet,level=lvl,create_plot=True);


The matrix profile can be calculated on the coefficients or on the reconstruction of each component:

In [ ]:
subseq_size = 20
quantile = 0.75
MPs, figs = get_MP_from_wavelets(yd,subseq_size=subseq_size,level=lvl,wavelet=wavelet, on_signals=True,quartile=quantile, create_plot=True, ignore_extremes=True, plot_data=True)
axs1 = figs[1]
max_y1 = np.max([np.abs(ax.get_ylim()) for ax in axs1])

MPs, figs = get_MP_from_wavelets(yd,subseq_size=subseq_size,level=lvl,wavelet=wavelet, on_signals=False,quartile=quantile, create_plot=True, ignore_extremes=True, plot_data=True)
axs2 = figs[1]
max_y2 = np.max([np.abs(ax.get_ylim()) for ax in axs2])

# Plot anomalies
for _,start,size,series in anomalies:
    for s in series:
        if s == sig:
            for ax in axs1:
                rect = Rectangle((start-subseq_size, -max_y1), size+subseq_size, 2*max_y1, facecolor='lightgrey')
                ax.add_patch(rect)
            for ax in axs2:
                rect = Rectangle((start-subseq_size, -max_y2), size+subseq_size, 2*max_y2, facecolor='lightgrey')
                ax.add_patch(rect)

In [ ]:
lvl = 5
k = 2
K = wavelet_KDP(yd,subseq_size,k,level=lvl,wavelet=wavelet,on_signals=True,quartile=quantile,create_plot=False,ignore_extremes=True,plot_data=False)
print_multiple(K);

K = wavelet_KDP(yd,subseq_size,k,level=lvl,wavelet=wavelet,on_signals=False,quartile=quantile,create_plot=False,ignore_extremes=True,plot_data=False)
print_multiple(K);

### Filtering

Before applying the stumpy procedure, a filtering technique is used. Here, the Butterworth pass-band filter from scipy is used.

In [ ]:
# Create filter
sampling = 1 # 1 measurement / hour
T_cutoff = np.array([30, 6]) # 6 hour cutoff period
filter_type = Filter_type.BUTTER_PASS
order = 2

cutoff_freq = 1/T_cutoff

num_coef, den_coef = get_filter(filter_type, sampling_freq=sampling, cutoff_freq=cutoff_freq, order=order)

w, h = sp.signal.freqz(num_coef, den_coef, fs=sampling)

plt.subplot(2, 1, 1)
plt.plot(w, np.abs(h), 'b')
plt.xlim(0, 0.5*sampling)
plt.title("Filter Frequency Response")
plt.xlabel('Frequency [1/h]')
plt.grid()


Then, we can apply the filter to the generated data

In [ ]:
filtering_on = True

if filtering_on:

    # Filtering signals
    y_filtered = apply_filter(y, num_coef, den_coef)

    # Plot graphs
    max_y = 2.5
    fig, axs = print_multiple(y_filtered, height=1, width=7, ylimits=[-max_y,max_y], ylabel='Filtered', xlabel='Timestamp')

    # Plot anomalies
    for _,start,size,series in anomalies:
        for s in series:
            rect = Rectangle((start, -max_y), size, 2*max_y, facecolor='lightgrey')
            axs[s].add_patch(rect)

    
else:
    y_filtered = y.copy()
    

### Matrix Profile

Now that we have the filtered data, we can calculate the Matrix Profile

In [ ]:
# Defining matrix profile subsequence size
sub_size = 30
quartile = 0.75

MPs = get_MP(y_filtered, sub_size, quartile = quartile, ignore_extremes= True)

In [ ]:
# Plot graphs
max_y = np.max([mp.iloc[:,0] for mp in MPs])
fig, axs = print_multiple(MPs, height=1, width=7, ylimits=[0,max_y], ylabel='MP', xlabel='Timestamp')

# Plot anomalies
for n,i,size,series in anomalies:
    for o in series:
        rect = Rectangle((i-sub_size, 0), sub_size+size, max_y, facecolor='lightgrey')
        axs[o].add_patch(rect)


In [ ]:
KDP = get_KDP(y_filtered, sub_size, quartile = quartile, ignore_start = True, pre_calc_MP=MPs)

# Plot graphs
fig, axs = print_multiple(KDP[:C,:], height=1, width=7, ylimits=[0,max_y], ylabel='KDP', xlabel='Timestamp')

for k in range(C):
    for n,i,size,series in anomalies:
        if C-k <= n:
            rect = Rectangle((i-sub_size, 0),sub_size+size, max_y, facecolor='lightgrey')
            axs[C-k-1].add_patch(rect)

### Animation

This subsection creates an animation that illustrates the MP using the first signal.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

# Re-calculating MP
sub_size = 30
quartile = 0.75

MPs = get_MP(y_filtered, sub_size, ignore_start = False)
closest_match = MPs[C,:]
num_intervals = MPs.shape[1]

# Reference subsequence start
start_blue = 100

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 150  
plt.ioff()


# Creating figure
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={'hspace': 0},figsize=[7,4])

# Plotting signal
axs[0].plot(y_filtered[0,:]);
axs[0].set_ylabel('Signal')
axs[0].grid(True)

# Creating blue rectangle around reference subsequence
rect1 = Rectangle((start_blue, -1.5), sub_size, 3.5, facecolor='lightblue')
axs[0].add_patch(rect1)

# Creating orange moving rectangle
j = 50
rect2 = Rectangle((0, -1.5), sub_size, 3.5, facecolor='orange')
axs[0].add_patch(rect2)
recs = [rect1, rect2]

# Red curve within orange rectangle
xline = np.arange(sub_size)
yline = y_filtered[0,start_blue:start_blue+sub_size]
line_moving, = axs[0].plot(xline,yline,'r--')

# Creating MP curve
axs[1].plot(MPs[0,:]);
axs[1].plot([start_blue],[MPs[0,start_blue]],'ro')
axs[1].set_xlabel('Timestamp')
axs[1].set_ylabel(f'Matrix Profile')
axs[1].grid(True)

# Animate function
time_stop = 5
time_anim = 3
fps = 30

stop_frames = fps*time_stop
anim_frames = fps*time_anim
frame_interval = 500

loc_frame = np.arange(0,num_intervals,num_intervals/anim_frames,dtype=float).astype(int)
def animate(frame):
    
    if frame < anim_frames:
        recs[1].set_xy([loc_frame[frame],-1.5])
        line_moving.set_xdata(xline+loc_frame[frame])
    else:
        recs[1].set_xy([closest_match[start_blue],-1.5])
        line_moving.set_xdata(xline+closest_match[start_blue])

    return [*recs,line_moving],

ani = animation.FuncAnimation(fig, animate, frames=stop_frames+anim_frames,interval=1000/fps)

ani.save('./anim.gif', writer='imagemagick', fps=fps)


In [ ]:
MPs